# RAGCheck quickstart

**pytest for RAG systems** — quality scores, failure taxonomy, judge validation, and CI regression checks for retrieval-augmented generation pipelines.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ekta-Shah/ragcheck/blob/main/examples/quickstart.ipynb)

This notebook: (1) a zero-API-key demo, (2) evaluating your own pipeline with a real LLM judge, (3) checking whether the judge deserves trust.

In [ ]:
%pip install -q git+https://github.com/Ekta-Shah/ragcheck  # PyPI: ragcheck-eval (soon)

## 1. The 10-second demo — no API key

A canned pipeline with two deliberately planted failures (a hallucinated claim and a confident answer to an unanswerable question) plus a deterministic offline judge, so you can see the whole workflow instantly. Real runs use Claude/Groq judges.

In [ ]:
!ragcheck demo

The HTML report shows the failing samples with their retrieved contexts inline:

In [ ]:
from IPython.display import HTML
HTML(open("ragcheck_output/demo.html").read())

## 2. Evaluate your own pipeline with a real judge

Any `(question: str) -> RAGResponse` callable works. Below: a toy keyword-retrieval pipeline over an in-memory corpus, generation + judging via Groq (free tier works — get a key at console.groq.com). Swap `GROQ_API_KEY` for `ANTHROPIC_API_KEY` and `judge_provider: anthropic` to use Claude.

In [ ]:
import os, getpass
os.environ["GROQ_API_KEY"] = getpass.getpass("Groq API key: ")

In [ ]:
from ragcheck import RAGResponse, RetrievedChunk
from ragcheck.adapters import FunctionAdapter
from ragcheck.llm import default_client

CORPUS = {
    "doc_leave": "Employees receive 24 days of paid leave per year, plus 8 public holidays.",
    "doc_wfh": "Remote work is allowed up to 3 days per week; Tuesdays are mandatory in-office.",
    "doc_notice": "The notice period is 60 days for all roles.",
}

llm = default_client()  # picks Groq/Anthropic from your env

def pipeline(question: str) -> RAGResponse:
    terms = {w.strip('?.,').lower() for w in question.split()}
    ranked = sorted(CORPUS.items(), key=lambda kv: -sum(t in kv[1].lower() for t in terms))
    chunks = [RetrievedChunk(content=text, source_id=doc) for doc, text in ranked[:2]]
    context = "\n".join(f"[{c.source_id}] {c.content}" for c in chunks)
    answer = llm.complete(
        f"Answer using ONLY these documents. If they lack the answer, say "
        f"'The provided documents do not contain this information.'\n\n{context}\n\nQ: {question}",
        max_tokens=128,
    ).text.strip()
    return RAGResponse(
        answer=answer,
        retrieved_chunks=chunks,
        refused="do not contain" in answer.lower(),
    )

adapter = FunctionAdapter(pipeline)

In [ ]:
from pathlib import Path
from ragcheck.config import EvalConfig, MetricSpec
from ragcheck.datasets.models import EvalDataset, QAPair
from ragcheck.runner import evaluate

dataset = EvalDataset(name="hr-toy", pairs=[
    QAPair(question="How many days of paid leave per year?", relevant_source_ids=["doc_leave"]),
    QAPair(question="How many remote days are allowed weekly?", relevant_source_ids=["doc_wfh"]),
    QAPair(question="What is the notice period?", relevant_source_ids=["doc_notice"]),
    QAPair(question="What is the parental leave policy?", answerable=False),
])

config = EvalConfig(
    dataset=Path("inline"), adapter="inline",
    metrics=[
        MetricSpec(name="hit_rate", params={"k": 2}),
        MetricSpec(name="faithfulness"),
        MetricSpec(name="refusal_calibration"),
    ],
    judge_provider="groq",
    output_dir=Path("ragcheck_output"), cache_path=Path("judge_cache.sqlite"),
    run_name="my-first-eval",
)

report, path = evaluate(adapter, dataset, config, adapter_name="toy-hr-pipeline")

from ragcheck.report.cli_summary import print_summary
print_summary(report)

Re-run the cell above: judgments come from the SQLite cache, so it's near-free. Every judged score records the judge model and prompt version.

## 3. Should you trust the judge?

Before relying on judged metrics, measure judge-vs-human agreement (Cohen's κ) on samples you label yourself:

In [ ]:
!wget -q https://raw.githubusercontent.com/Ekta-Shah/ragcheck/main/examples/judge_labels_sample.jsonl
!ragcheck validate-judge judge_labels_sample.jsonl --metric faithfulness \
    --provider groq --threshold 1.0

Try `--threshold 0.5` and watch κ drop — the validation tells you the judge's safe operating point.

## Next steps

- [Quickstart docs](https://github.com/Ekta-Shah/ragcheck/blob/main/docs/quickstart.md) — YAML configs + the `ragcheck run` CLI
- [Metrics reference](https://github.com/Ekta-Shah/ragcheck/blob/main/docs/metrics.md) — all 9 metrics, what they measure, limitations
- [CI integration](https://github.com/Ekta-Shah/ragcheck/blob/main/docs/ci-integration.md) — `ragcheck compare --fail-if` to gate PRs
- `ragcheck generate-dataset your_docs/` — build an eval set from any folder of text/markdown

⭐ If this was useful: [github.com/Ekta-Shah/ragcheck](https://github.com/Ekta-Shah/ragcheck)